In [ ]:
import pandas as pd
import numpy as np
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch
from sklearn.metrics import roc_auc_score
import random
import glob

In [ ]:
all_files = glob.glob(os.path.join('../patients_for_spaces_checks/', "*.csv"))

if not all_files:
    print(f"No CSV files found in the directory: {directory_path}")

list_of_dfs = []
for file_path in all_files:
    try:
        df = pd.read_csv(file_path)
        list_of_dfs.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        continue

if list_of_dfs:
    combined_df = pd.concat(list_of_dfs, ignore_index=True)
else:
    print("No DataFrames could be successfully loaded.")


In [ ]:
validation_set = combined_df
validation_set.info()


In [ ]:
validation_set.info()

In [ ]:
validation_set.this_space.nunique()

In [ ]:
validation_set.patient_summary.nunique()

In [ ]:
# precision and MAP at 20, with no roberta check

import pandas as pd
import numpy as np


def average_precision(label_array):
    total_yes = np.sum(label_array)
    if total_yes > 0:
        yes_indices = np.where(label_array == 1)[0] + 1
        precisions = []
        for index in yes_indices:
            precision = np.sum(label_array[0:index])/index
            precisions.append(precision)
        precisions = np.sum(np.array(precisions))
        return precisions / total_yes
    else:
        return 0

print(validation_set.info())

print(validation_set.eligibility_result.value_counts()/validation_set.shape[0])
temp = validation_set.groupby('this_space').head(40)
temp = validation_set.groupby('this_space').eligibility_result.apply(average_precision)
mapk = np.sum(temp)/len(temp)
print('map @ 40 of the reranker_round2.model')
print(mapk)

In [ ]:
# now enrich with trial checker

In [ ]:
validation_set['pt_trial_pair'] = validation_set['this_space'] + "\nNow here is the patient summary:" + validation_set['patient_summary']
validation_set = validation_set[~validation_set.pt_trial_pair.isnull()]

In [ ]:
from transformers import pipeline, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('../../../v20_models/trialchecker')
pipe = pipeline('text-classification', '../../../v20_models/trialchecker', tokenizer=tokenizer, truncation=True, padding='max_length', max_length=2048, device='cuda') 
predictions = pipe(validation_set.pt_trial_pair.tolist())

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
predictions_frame = pd.DataFrame(predictions)
predictions_frame['score'] = np.where(predictions_frame.label=='NEGATIVE', 1 - predictions_frame.score, predictions_frame.score)
predictions_frame['logit_score'] = np.log(predictions_frame.score / (1 - predictions_frame.score))


In [ ]:
validation_set.shape[0]

In [ ]:
predictions_frame.logit_score.isnull().value_counts()

In [ ]:
pruned_set = pd.concat([validation_set.reset_index(), predictions_frame.reset_index()], axis=1)

In [ ]:
pruned_set.info()
pruned_set.to_csv('trial_centric_with_trial_checker_responses.csv')

In [ ]:
import pandas as pd
pruned_set = pd.read_csv('trial_centric_with_trial_checker_responses.csv')

In [ ]:
# checker results on round 1 generated data ('round 2 data')
from sklearn.metrics import roc_auc_score
roc_auc_score(pruned_set.eligibility_result, pruned_set.score)

In [ ]:
from utils_102023 import eval_model
eval_model(pruned_set.score, pruned_set.eligibility_result, graph=True)

In [ ]:
import numpy as np
from sklearn.metrics import cohen_kappa_score
cohen_kappa_score(np.where(pruned_set.eligibility_result == 0.0, 'NEGATIVE','POSITIVE'), pruned_set.label)

In [ ]:
# if next line were removed it would pull more than 20 original checks and later filter for the top 20
pruned_set = pruned_set.reset_index().groupby('this_space').head(40)
pruned_set = pruned_set[pruned_set.label == 'POSITIVE']


In [ ]:
pruned_set.info()

In [ ]:
# median number of results returned after trial checker
pruned_set.groupby('this_space').size().median()

In [ ]:
# median number of results returned after trial checker
pruned_set.groupby('this_space').size().mean()

In [ ]:
pruned_set.patient_summary.nunique()

In [ ]:
pruned_set.this_space.nunique()

In [ ]:

def average_precision(label_array):
    total_yes = np.sum(label_array)
    if total_yes > 0:
        yes_indices = np.where(label_array == 1)[0] + 1
        precisions = []
        for index in yes_indices:
            precision = np.sum(label_array[0:index])/index
            precisions.append(precision)
        precisions = np.sum(np.array(precisions))
        return precisions / total_yes
    else:
        return 0

print(pruned_set.info())

print(pruned_set.eligibility_result.value_counts()/pruned_set.shape[0])
temp = pruned_set.groupby('this_space').head(40)
print(temp.eligibility_result.value_counts()/temp.shape[0])
temp = pruned_set.groupby('this_space').eligibility_result.apply(average_precision)
mapk = np.sum(temp)/len(temp)
print('map @ 40 of the round2.model on the external patients for trials task with llama check')
print(mapk)